# 타이타닉 생존 분석: 시각화와 예측 모델

앞에서 만든 품질 점검에 이어서, 이번에는 **시각화**로 데이터를 이해하고 **머신러닝 모델**로 생존 여부를 예측합니다.

순서:
1. 데이터 불러오기 및 정제
2. 시각화 (범주형 / 수치형 / 상관관계)
3. 학습/테스트 데이터 분리
4. 모델 3종 학습 (로지스틱 회귀, 결정트리, 랜덤포레스트)
5. 성능 평가 (정확도, 혼동행렬, ROC)
6. 변수 중요도

## 0. 라이브러리 불러오기

In [ ]:
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns

from sklearn.ensemble import RandomForestClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (
    ConfusionMatrixDisplay,
    accuracy_score,
    classification_report,
    roc_auc_score,
    roc_curve,
)
from sklearn.model_selection import train_test_split
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.tree import DecisionTreeClassifier

plt.rcParams["font.family"] = "AppleGothic"
plt.rcParams["axes.unicode_minus"] = False
sns.set_theme(style="whitegrid", font="AppleGothic")

## 1. 데이터 불러오기 및 정제

- 정보가 없는 `zero.*` 컬럼 19개를 제거합니다.
- `Passengerid` 는 단순 번호라 제거합니다.
- `Embarked` 결측치 2개는 최빈값으로 채웁니다.
- `2urvived` 는 `Survived` 로 이름을 바꿉니다.

In [ ]:
DATA_PATH = "/Users/remchoi/.cache/kagglehub/datasets/heptapod/titanic/versions/1/train_and_test2.csv"

df = pd.read_csv(DATA_PATH).rename(columns={"2urvived": "Survived"})

zero_cols = [c for c in df.columns if c.startswith("zero")]
df = df.drop(columns=zero_cols + ["Passengerid"])

df["Embarked"] = df["Embarked"].fillna(df["Embarked"].mode()[0])

print("정제 후 컬럼:", df.columns.tolist())
print("행/열:", df.shape)
df.head()

## 2. 시각화 - 범주형 변수

성별(Sex), 객실등급(Pclass), 탑승항구(Embarked)에 따라 생존율이 어떻게 다른지 봅니다.
`hue="Survived"` 를 주면 생존 여부별로 막대가 나뉩니다.

In [ ]:
cat_cols = ["Sex", "Pclass", "Embarked"]

fig, axes = plt.subplots(1, 3, figsize=(16, 5))
for ax, col in zip(axes, cat_cols):
    sns.countplot(data=df, x=col, hue="Survived", ax=ax, palette="Set2")
    ax.set_title(f"{col} 별 생존자 수")
    ax.legend(title="Survived", labels=["사망(0)", "생존(1)"])
plt.tight_layout()
plt.show()

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(16, 5))
for ax, col in zip(axes, cat_cols):
    rate = df.groupby(col)["Survived"].mean()
    rate.plot(kind="bar", ax=ax, color="steelblue")
    ax.set_title(f"{col} 별 생존율")
    ax.set_ylabel("생존율")
    ax.set_ylim(0, 1)
    for i, v in enumerate(rate):
        ax.text(i, v + 0.02, f"{v:.2f}", ha="center")
plt.tight_layout()
plt.show()

## 3. 시각화 - 수치형 변수

나이(Age)와 운임(Fare)의 분포를 생존 여부별로 비교합니다.
`hue="Survived"` 로 겹쳐 그리면 차이가 잘 보입니다.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))
sns.histplot(data=df, x="Age", hue="Survived", kde=True, bins=30, ax=axes[0], palette="Set1")
axes[0].set_title("Age 분포 (생존 여부별)")
sns.histplot(data=df, x="Fare", hue="Survived", kde=True, bins=30, ax=axes[1], palette="Set1")
axes[1].set_title("Fare 분포 (생존 여부별)")
plt.tight_layout()
plt.show()

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))
sns.boxplot(data=df, x="Survived", y="Age", ax=axes[0], hue="Survived", palette="Set2", legend=False)
axes[0].set_title("Survived 별 Age")
sns.boxplot(data=df, x="Survived", y="Fare", ax=axes[1], hue="Survived", palette="Set2", legend=False)
axes[1].set_title("Survived 별 Fare")
plt.tight_layout()
plt.show()

## 4. 시각화 - 상관관계

`corr()` 로 변수들 간 상관계수를 구해 히트맵으로 봅니다.
값이 1에 가까우면 양의 상관, -1에 가까우면 음의 상관입니다.

In [ ]:
plt.figure(figsize=(8, 6))
sns.heatmap(df.corr(numeric_only=True), annot=True, fmt=".2f", cmap="coolwarm", center=0)
plt.title("변수 간 상관관계")
plt.show()

## 5. 학습/테스트 데이터 분리

- `X`: 정답(Survived)을 제외한 입력 변수
- `y`: 정답(Survived)
- `train_test_split`: 80% 는 학습, 20% 는 평가용. `stratify=y` 로 생존 비율을 유지합니다.

In [ ]:
X = df.drop(columns="Survived")
y = df["Survived"]

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

print("학습 데이터:", X_train.shape)
print("테스트 데이터:", X_test.shape)

## 6. 모델 3종 학습

- **로지스틱 회귀**: 가장 기본적인 분류 모델 (스케일링 파이프라인 사용)
- **결정트리**: 조건 분기로 예측, 해석이 쉬움
- **랜덤포레스트**: 결정트리 여러 개를 합쳐 성능을 높인 모델

In [ ]:
models = {
    "LogisticRegression": make_pipeline(StandardScaler(), LogisticRegression(max_iter=1000)),
    "DecisionTree": DecisionTreeClassifier(max_depth=4, random_state=42),
    "RandomForest": RandomForestClassifier(n_estimators=200, max_depth=5, random_state=42),
}

results = {}
for name, model in models.items():
    model.fit(X_train, y_train)
    pred = model.predict(X_test)
    proba = model.predict_proba(X_test)[:, 1]
    results[name] = {
        "model": model,
        "accuracy": accuracy_score(y_test, pred),
        "auc": roc_auc_score(y_test, proba),
        "pred": pred,
        "proba": proba,
    }
    print(f"{name:20s} 정확도={results[name]['accuracy']:.3f}  AUC={results[name]['auc']:.3f}")

## 7. 모델 평가 - 혼동행렬과 분류 리포트

In [ ]:
best_name = max(results, key=lambda k: results[k]["auc"])
print("최고 모델(AUC 기준):", best_name)

best = results[best_name]
ConfusionMatrixDisplay.from_predictions(y_test, best["pred"], display_labels=["사망", "생존"])
plt.title(f"{best_name} 혼동행렬")
plt.show()

print(classification_report(y_test, best["pred"], target_names=["사망", "생존"]))

## 8. ROC 곡선

ROC 곡선은 분류 성능을 나타내는 그래프입니다. 곡선이 왼쪽 위에 가까울수록 좋고,
아래 면적(AUC)이 1에 가까울수록 우수합니다.

In [ ]:
plt.figure(figsize=(7, 6))
for name, r in results.items():
    fpr, tpr, _ = roc_curve(y_test, r["proba"])
    plt.plot(fpr, tpr, label=f"{name} (AUC={r['auc']:.3f})")
plt.plot([0, 1], [0, 1], "k--", label="무작위 예측")
plt.xlabel("거짓 양성률 (FPR)")
plt.ylabel("참 양성률 (TPR)")
plt.title("ROC 곡선")
plt.legend()
plt.show()

## 9. 변수 중요도 (RandomForest)

In [ ]:
rf = results["RandomForest"]["model"]
importance = pd.Series(rf.feature_importances_, index=X.columns).sort_values()

importance.plot(kind="barh", figsize=(8, 5), color="mediumseagreen")
plt.title("변수 중요도")
plt.xlabel("중요도")
plt.tight_layout()
plt.show()

## 정리

- 성별(`Sex`), 객실등급(`Pclass`), 운임(`Fare`)이 생존에 큰 영향을 주는 것으로 보입니다.
- 세 모델 중 AUC와 정확도를 비교해 최종 모델을 고릅니다.
- 이 데이터는 원본 타이타닉과 달리 `Age` 결측치가 없고, `zero.*` 컬럼이 섞여 있어 정제가 중요했습니다.

### 다음 단계 아이디어
1. 하이퍼파라미터 튜닝 (`GridSearchCV`)
2. 교차검증 (`cross_val_score`)으로 더 안정적인 성능 확인
3. `FamilySize = sibsp + Parch + 1` 같은 파생 변수 만들기